In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.5 FlashAttention

The attention we built materializes the full `T x T` score matrix. **FlashAttention**
computes the same result in tiles without ever storing that matrix, saving memory
and time. It is the same math, so the outputs match, we teach with the explicit
version and ship the fused one.

In [ ]:
from torch.nn import functional as F
from pocketlm import scaled_dot_product_attention

torch.manual_seed(0)
q = torch.randn(1, 4, 6, 8)
k = torch.randn(1, 4, 6, 8)
v = torch.randn(1, 4, 6, 8)
ours, _ = scaled_dot_product_attention(q, k, v, causal=True)
fused = F.scaled_dot_product_attention(q, k, v, is_causal=True)  # fused kernel
print("max difference:", (ours - fused).abs().max().item())

The two agree to floating-point tolerance. In production you call the fused path;
in this course we keep the explicit one because you can read it.

In [ ]:
assert torch.allclose(ours, fused, atol=1e-5)